# TF-IDF vs BM25 на корпусе поддержки интернет-магазина

Перед каждой ячейкой с кодом — «🔮 Предскажи». Сначала пишешь предсказание в markdown-ячейке, потом запускаешь.

In [ ]:
from retrieval import BM25, TfidfRetriever, tokenize

CORPUS = [
    "Ошибка 0x80070057 при обновлении Windows: как исправить",
    "Процедура возврата средств: оформление заявки и сроки",
    "Возврат возможен только при наличии чека и упаковки",
    "Доставка курьером по Москве занимает один-два рабочих дня",
    "Самовывоз заказа из пункта выдачи: доставка бесплатная, хранение семь дней",
    "Оплата заказа картой онлайн, наличными или через СБП при получении",
    "Если оплата не прошла, деньги вернутся на карту в течение трёх дней",
    "Гарантия на технику один год, ремонт проводится в сервисном центре",
    "Гарантийный ремонт: как сдать товар и сколько ждать",
    (
        "Часто задаваемые вопросы покупателей. Возврат товара надлежащего качества возможен "
        "в течение 14 дней. Возврат товара с дефектом оформляется по гарантии. Возврат техники "
        "требует проверки в сервисном центре. Возврат одежды и обуви принимается при сохранённых "
        "ярлыках. Доставка возвращаемого товара оплачивается покупателем, кроме случаев брака. "
        "Возврат денег на карту занимает до десяти рабочих дней. Возврат наличными производится "
        "в пункте выдачи. Подарочные сертификаты и бонусные баллы не подлежат обмену. Возврат "
        "цифровых товаров невозможен после активации ключа. Если возврат отклонён, напишите в "
        "поддержку, приложите фото и номер заказа, и оператор свяжется с вами в течение суток."
    ),
]
WINDOWS, PROCEDURE, SHORT, LONG = 0, 1, 2, 9

for i, d in enumerate(CORPUS):
    toks = tokenize(d)
    print(f"[{i}] токенов={len(toks):3d}  'возврат'×{toks.count('возврат')}  {d[:70]}")

In [ ]:
def show(name, results):
    print(name)
    for doc_id, score in results:
        print(f"  [{doc_id}] {score:.4f}  {CORPUS[doc_id][:60]}")

tfidf = TfidfRetriever().fit(CORPUS)
bm25 = BM25().fit(CORPUS)

## 1. Редкий точный токен

🔮 **Предскажи:** запрос `"0x80070057"` — какой документ будет топ-1 у TF-IDF и у BM25? Что будет на местах 2–3 и с какими скорами?

**Моё предсказание:** …

In [ ]:
q = "0x80070057"
show("TF-IDF", tfidf.search(q, k=3))
show("BM25", bm25.search(q, k=3))

## 2. Синонимы и словоформы

🔮 **Предскажи:** запрос `"как вернуть деньги за заказ"` — найдёт ли TF-IDF или BM25 документ [1] «Процедура возврата средств»? Какой у него будет скор? Что окажется в топе вместо него?

**Моё предсказание:** …

In [ ]:
q = "как вернуть деньги за заказ"
show("TF-IDF", tfidf.search(q, k=3))
show("BM25", bm25.search(q, k=3))
print()
print("скор документа [1]:  TF-IDF =", tfidf.transform([q])[0] @ tfidf.doc_matrix_[PROCEDURE],
      "  BM25 =", bm25.score(q)[PROCEDURE])
print("токены запроса:  ", tokenize(q))
print("токены документа:", tokenize(CORPUS[PROCEDURE]))

❓ **Почему так и что бы это починило?** (ответ — в чате)

## 3. Короткий документ с tf=1 против длинного с tf=8

🔮 **Предскажи:** запрос `"возврат"`, BM25 с `k1=1.5, b=0.75` — кто выше: короткий [2] (tf=1) или длинный [9] (tf=8)? Примерно во сколько раз отличаются скоры?

**Моё предсказание:** …

In [ ]:
def short_vs_long(s):
    print(f"короткий [2]: {s[SHORT]:.4f}   длинный [9]: {s[LONG]:.4f}   длинный/короткий = {s[LONG] / s[SHORT]:.2f}")

short_vs_long(BM25(k1=1.5, b=0.75).fit(CORPUS).score("возврат"))

🔮 **Предскажи:** то же, но `b=0`. Что изменится и почему?

**Моё предсказание:** …

In [ ]:
short_vs_long(BM25(k1=1.5, b=0).fit(CORPUS).score("возврат"))

🔮 **Предскажи:** то же, но `k1=0` (b=0.75). Что изменится и почему?

**Моё предсказание:** …

In [ ]:
short_vs_long(BM25(k1=0, b=0.75).fit(CORPUS).score("возврат"))

🔮 **Предскажи:** то же у TF-IDF (косинус запроса с документом). Кто выше и почему?

**Моё предсказание:** …

In [ ]:
short_vs_long(tfidf.transform(["возврат"])[0] @ tfidf.doc_matrix_.T)

## 4. Насыщение по tf

🔮 **Предскажи:** как будет выглядеть вклад одного термина в зависимости от tf (0…20) при `|d| = avgdl`, `idf = 1` для `k1` ∈ {0, 0.5, 1.5, 3}? Какая кривая выше всех при tf=20? Как выглядит кривая при `k1=0`?

**Моё предсказание:** …

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tf = np.arange(0, 21)
k1_values = [0, 0.5, 1.5, 3]
colors = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]  # один оттенок: светлее = меньше k1

fig, ax = plt.subplots(figsize=(7, 4))
for k1, c in zip(k1_values, colors):
    # |d| = avgdl -> (1 - b + b * |d| / avgdl) = 1, idf = 1
    contrib = np.divide(tf * (k1 + 1), tf + k1, out=np.zeros(len(tf)), where=tf > 0)
    ax.plot(tf, contrib, color=c, linewidth=2, marker="o", markersize=4, label=f"k1 = {k1}")
    ax.annotate(f"k1={k1}", (tf[-1], contrib[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9, color="#444")

ax.set_xlabel("tf — сколько раз термин встретился в документе")
ax.set_ylabel("вклад термина в BM25")
ax.set_title("Насыщение по tf (|d| = avgdl, idf = 1)")
ax.grid(alpha=0.25)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_xlim(0, 22.5)
ax.set_xticks(range(0, 21, 2))
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

❓ **К какому значению стремится вклад при tf → ∞ и почему?** (ответ — в чате)